In [38]:
import pandas as pd
import re

# ==============================
# 1. Đọc file
# ==============================
df = pd.read_excel("data tổng hợp.xlsx")

print("Số dòng ban đầu:", len(df))
print("Các cột:", df.columns)

# ==============================
# 2. Xác định cột
# ==============================
# Giả sử file bạn có 2 cột: sentiment + nội dung
content_col = None

for col in df.columns:
    if col.lower() in ['text', 'content', 'message', 'post', 'caption', 'sentence']:
        content_col = col
        break

if content_col is None:
    raise Exception("Không tìm thấy cột nội dung!")

# ==============================
# 3. Làm sạch dữ liệu
# ==============================
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    # xóa link
    text = re.sub(r'http\S+|www\S+', '', text)

    # xóa ký tự đặc biệt
    text = re.sub(r'[^a-zA-ZÀ-ỹ0-9\s]', ' ', text)

    # xóa khoảng trắng dư
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['sentence'] = df[content_col].apply(clean_text)

# ==============================
# 4. Lọc HUTECH
# ==============================
hutech_keywords = ["hutech", "đại học công nghệ tp hcm", "dh hutech"]

other_universities = [
    "bách khoa", "ngoại thương", "kinh tế", "y dược",
    "công nghiệp", "fpt", "huflit", "uit", "ussh",
    "ueh", "hcmut", "hcmus"
]

def filter_hutech(text):
    if any(k in text for k in hutech_keywords):
        return True
    if any(k in text for k in other_universities):
        return False
    return True

df = df[df['sentence'].apply(filter_hutech)]

print("Sau lọc HUTECH:", len(df))

# ==============================
# 5. Xóa rỗng
# ==============================
df = df[df['sentence'] != ""]

# ==============================
# 6. Xóa trùng
# ==============================
before = len(df)

df = df.drop_duplicates(subset=['sentence'])

after = len(df)

print("Đã xóa trùng:", before - after)

# ==============================
# 7. Reset index
# ==============================
df = df.reset_index(drop=True)

# ==============================
# 8. Lưu file (GIỮ 2 CỘT)
# ==============================
df_final = df[['sentiment', 'sentence']]

df_final.to_excel("data_cleaned.xlsx", index=False)

print("Hoàn thành!")
print("Số dòng cuối:", len(df_final))

Số dòng ban đầu: 19612
Các cột: Index(['sentiment', 'sentence'], dtype='object')
Sau lọc HUTECH: 19558
Đã xóa trùng: 423
Hoàn thành!
Số dòng cuối: 19135
